In [ ]:
import torch

from cfg_tools import load_config_files

from numb_project.constants import CONFIG_FOLDER, GW_CHECKPOINT_FOLDER
from numb_project.domain_module import get_domains_config
from numb_project.gw_model import MyGlobalWorkspace, get_global_workspace_mods
from numb_project.data_module import MnistDataModule, rotate_image_batch


config = load_config_files(
    CONFIG_FOLDER,
    use_cli=False,
    load_files=["config.yaml"])[0]

domain_configs = get_domains_config(['image', 'digit'])

gw_mod, selection_mod, operation_mod, operation_selection_mod, task_mod, loss_mod, operation_rotate_pixel_mod, operation_rotate_gw_mod = get_global_workspace_mods(config, domain_configs)

gw = MyGlobalWorkspace.load_from_checkpoint(
    f"{GW_CHECKPOINT_FOLDER}/pretrain.ckpt",
    gw_mod=gw_mod,
    selection_mod=selection_mod,
    loss_mod=loss_mod,
    weights_only=False
    )

data_module = MnistDataModule(50)

mnist_vae = gw.gw_mod.domain_mods.image.vae
image_encoder = gw.gw_mod.gw_encoders.image
digit_decoder = gw.gw_mod.gw_decoders.digit

mnist_vae.eval()
image_encoder.eval()
digit_decoder.eval()

device = next(mnist_vae.parameters()).device

# Récupère un batch avec image ET digit (le domaine combiné)
batch, batch_idx, dataloader_idx = next(iter(data_module.test_dataloader()))

upright_image_batch, digit_batch = None, None
for key, value in batch.items():
    if "image" in value and "digit" in value:
        upright_image_batch = value["image"].to(device)
        digit_batch = value["digit"].to(device)
        break

if upright_image_batch is None:
    raise ValueError("Aucune clé du batch ne contient à la fois 'image' et 'digit'")

true_digits = digit_batch.argmax(dim=1)
batch_size = upright_image_batch.shape[0]

# Rotation aléatoire initiale de chaque image du batch, pour voir si le
# modèle arrive à retrouver le bon digit malgré une orientation quelconque.
initial_angle = torch.rand(batch_size, device=device) * 360.0
image_batch = rotate_image_batch(upright_image_batch, initial_angle)

image_batch.shape

In [ ]:
import torch
import torch.nn.functional as F

from numb_project.data_module import rotate_image_batch

# Pour chaque image du batch (déjà tournée aléatoirement) : prédit le digit
# à 0°, tourne de 20° en 20° jusqu'à avoir fait un tour complet, et garde
# la prédiction dont la confiance (probabilité softmax du digit prédit)
# est la plus proche de 1.
angles = torch.arange(0, 360, 20, device=device, dtype=torch.float32)

best_confidence = torch.full((batch_size,), -1.0, device=device)
best_digit = torch.zeros(batch_size, dtype=torch.long, device=device)
best_angle = torch.zeros(batch_size, device=device)

with torch.no_grad():
    for angle in angles:
        angle_deg = torch.full((batch_size,), angle.item(), device=device)
        rotated = rotate_image_batch(image_batch, angle_deg)

        mu, _ = mnist_vae.encoder(rotated)
        gw_latent = image_encoder(mu)
        digit_logits = digit_decoder(gw_latent)
        confidence, digit_pred = F.softmax(digit_logits, dim=1).max(dim=1)

        improved = confidence > best_confidence
        best_confidence[improved] = confidence[improved]
        best_digit[improved] = digit_pred[improved]
        best_angle[improved] = angle

correct = best_digit == true_digits
print(f"Accuracy (digit correct après recherche de rotation): {correct.float().mean().item():.2%}\n")

for i in range(batch_size):
    status = "✓" if correct[i] else "✗"
    print(
        f"{status} Image {i}: digit vrai={true_digits[i].item()}, prédit={best_digit[i].item()}, "
        f"confiance={best_confidence[i].item():.4f}, "
        f"rotation initiale={initial_angle[i].item():.0f}°, angle retenu={best_angle[i].item():.0f}°"
    )

In [ ]:
import matplotlib.pyplot as plt

# Affichage : image originale (upright), image tournée aléatoirement (entrée
# du modèle), image redressée à l'angle retenu, et prédiction
n_show = min(8, batch_size)

fig, axes = plt.subplots(3, n_show, figsize=(n_show * 1.5, 4.5))
for i in range(n_show):
    straightened = rotate_image_batch(image_batch[i : i + 1], best_angle[i : i + 1])[0, 0].cpu().numpy()

    axes[0, i].imshow(upright_image_batch[i, 0].cpu().numpy(), cmap="gray")
    axes[0, i].axis("off")
    axes[0, i].set_title(f"original ({true_digits[i].item()})", fontsize=8)

    axes[1, i].imshow(image_batch[i, 0].cpu().numpy(), cmap="gray")
    axes[1, i].axis("off")
    axes[1, i].set_title(f"entrée ({initial_angle[i].item():.0f}°)", fontsize=8)

    axes[2, i].imshow(straightened, cmap="gray")
    axes[2, i].axis("off")
    axes[2, i].set_title(
        f"+{best_angle[i].item():.0f}° -> {best_digit[i].item()}\n(conf={best_confidence[i].item():.2f})",
        fontsize=8,
    )

plt.tight_layout()
plt.show()

In [ ]:
accuracy = (best_digit == true_digits).float().mean().item()
print(f"Précision globale (accuracy) sur le batch: {accuracy:.2%} ({(best_digit == true_digits).sum().item()}/{batch_size})\n")

print("Précision par digit (TP / nb de fois prédit):")
for digit in range(10):
    predicted_as_digit = best_digit == digit
    n_predicted = predicted_as_digit.sum().item()
    if n_predicted == 0:
        print(f"  digit {digit}: jamais prédit")
        continue
    true_positives = (predicted_as_digit & (true_digits == digit)).sum().item()
    precision = true_positives / n_predicted
    print(f"  digit {digit}: {precision:.2%} ({true_positives}/{n_predicted})")

In [ ]:
# Angle attendu pour redresser l'image (celui qui annule la rotation initiale),
# et écart circulaire entre cet angle et l'angle retenu par la recherche.
expected_angle = (360.0 - initial_angle) % 360.0
angle_diff = (best_angle - expected_angle) % 360.0
angle_error = torch.minimum(angle_diff, 360.0 - angle_diff)

for label, mask in [("Bonnes prédictions", correct), ("Mauvaises prédictions", ~correct)]:
    n = mask.sum().item()
    if n == 0:
        print(f"{label}: aucune image")
        continue
    errors = angle_error[mask]
    print(
        f"{label} ({n} images): erreur d'angle moyenne={errors.mean().item():.1f}°, "
        f"médiane={errors.median().item():.1f}°, min={errors.min().item():.1f}°, max={errors.max().item():.1f}°"
    )

print()
for i in range(batch_size):
    status = "✓" if correct[i] else "✗"
    print(
        f"{status} Image {i}: angle attendu={expected_angle[i].item():.0f}°, "
        f"angle retenu={best_angle[i].item():.0f}°, erreur={angle_error[i].item():.0f}°"
    )

## Boucle perception → global workspace → action, apprise par renforcement (REINFORCE)

Au lieu de balayer tous les angles, un agent observe l'image courante (perception → GW),
choisit une petite rotation à appliquer (l'action, prise dans un environnement traité comme
une boîte noire non différentiable, comme le ferait un robot), puis recommence sur la nouvelle
image obtenue. La récompense à chaque pas est la confiance de lecture du digit dans la GW
(`max(softmax(digit_decoder(state)))`) — aucun vrai label digit n'intervient dans l'apprentissage,
seulement dans les métriques affichées pour évaluer le progrès.

In [ ]:
import torch.nn as nn
from torch.distributions import Categorical

# Ensemble d'actions discrètes : ne pas bouger, ou tourner de 20°
ACTION_DEGREES = torch.tensor([0.0, 20.0], device=device)
N_ACTIONS = len(ACTION_DEGREES)
N_STEPS = 8  # nombre de pas perception -> action par épisode

train_loader = data_module.train_dataloader()


def get_image_digit_batch(loader) -> tuple[torch.Tensor, torch.Tensor]:
    batch, _, _ = next(iter(loader))
    for key, value in batch.items():
        if "image" in value and "digit" in value:
            return value["image"].to(device), value["digit"].to(device)
    raise ValueError("Aucune clé du batch ne contient à la fois 'image' et 'digit'")


def perceive(images: torch.Tensor) -> torch.Tensor:
    """Perception : image -> latent VAE -> état de l'espace global (GW)."""
    with torch.no_grad():
        mu, _ = mnist_vae.encoder(images)
        return image_encoder(mu)


def confidence_of(state: torch.Tensor) -> torch.Tensor:
    """Récompense non supervisée : confiance de lecture du digit dans la GW."""
    with torch.no_grad():
        probs = F.softmax(digit_decoder(state), dim=1)
        return probs.max(dim=1).values


class RotationPolicy(nn.Module):
    def __init__(self, state_dim: int, n_actions: int, hidden_dim: int = 64) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_actions),
        )

    def forward(self, state: torch.Tensor) -> torch.Tensor:
        return torch.softmax(self.net(state), dim=-1)


state_dim = perceive(upright_image_batch[:1]).shape[-1]
policy = RotationPolicy(state_dim, N_ACTIONS).to(device)
optimizer = torch.optim.Adam(policy.parameters(), lr=1e-3)

In [ ]:
n_episodes = 400
gamma = 0.9

reward_history = []
accuracy_history = []

policy.train()
for episode in range(n_episodes):
    # Nouveau batch, tourné aléatoirement au départ : l'agent découvre l'image
    # dans une orientation quelconque, comme un objet posé devant un robot.
    ep_upright, ep_digit = get_image_digit_batch(train_loader)
    ep_true_digits = ep_digit.argmax(dim=1)
    b = ep_upright.shape[0]

    cumulative_angle = torch.rand(b, device=device) * 360.0
    current_image = rotate_image_batch(ep_upright, cumulative_angle)
    state = perceive(current_image)

    log_probs, rewards = [], []
    for step in range(N_STEPS):
        action_probs = policy(state)
        dist = Categorical(action_probs)
        action_idx = dist.sample()
        log_probs.append(dist.log_prob(action_idx))

        # Action : applique la rotation choisie sur l'environnement (l'image)
        cumulative_angle = (cumulative_angle + ACTION_DEGREES[action_idx]) % 360.0
        current_image = rotate_image_batch(ep_upright, cumulative_angle)
        state = perceive(current_image)

        rewards.append(confidence_of(state))

    rewards = torch.stack(rewards)  # [N_STEPS, B]
    log_probs = torch.stack(log_probs)  # [N_STEPS, B]

    # Retours actualisés (retour-to-go), avec baseline = moyenne du batch à
    # chaque pas pour réduire la variance du gradient de politique.
    returns = torch.zeros_like(rewards)
    running = torch.zeros(b, device=device)
    for t in reversed(range(N_STEPS)):
        running = rewards[t] + gamma * running
        returns[t] = running
    advantage = returns - returns.mean(dim=1, keepdim=True)

    loss = -(log_probs * advantage.detach()).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    with torch.no_grad():
        final_pred = digit_decoder(state).argmax(dim=1)
        acc = (final_pred == ep_true_digits).float().mean().item()

    reward_history.append(rewards.mean().item())
    accuracy_history.append(acc)

    if episode % 25 == 0 or episode == n_episodes - 1:
        print(
            f"épisode {episode:4d}: récompense moyenne={reward_history[-1]:.3f}, "
            f"accuracy finale (éval seule, non utilisée pour l'entraînement)={acc:.2%}"
        )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(reward_history)
axes[0].set_title("Récompense moyenne (confiance) par épisode")
axes[0].set_xlabel("épisode")

axes[1].plot(accuracy_history)
axes[1].set_title("Accuracy finale par épisode (éval seule)")
axes[1].set_xlabel("épisode")

plt.tight_layout()
plt.show()

In [ ]:
# Évaluation : politique déterministe (action la plus probable) sur un batch
# de test frais, en suivant toute la trajectoire pas à pas.
policy.eval()

eval_upright, eval_digit = get_image_digit_batch(data_module.test_dataloader())
eval_true_digits = eval_digit.argmax(dim=1)
eval_b = eval_upright.shape[0]

eval_angle = torch.rand(eval_b, device=device) * 360.0
trajectory_images = [rotate_image_batch(eval_upright, eval_angle)]

state = perceive(trajectory_images[0])
with torch.no_grad():
    # Récompense attendue sur l'image de départ, avant toute action
    probs = F.softmax(digit_decoder(state), dim=1)
    confidence, digit_pred = probs.max(dim=1)
trajectory_confidences = [confidence]
trajectory_digits = [digit_pred]

with torch.no_grad():
    for step in range(N_STEPS):
        action_idx = policy(state).argmax(dim=-1)
        eval_angle = (eval_angle + ACTION_DEGREES[action_idx]) % 360.0

        current_image = rotate_image_batch(eval_upright, eval_angle)
        state = perceive(current_image)

        probs = F.softmax(digit_decoder(state), dim=1)
        confidence, digit_pred = probs.max(dim=1)

        trajectory_images.append(current_image)
        trajectory_confidences.append(confidence)
        trajectory_digits.append(digit_pred)

eval_correct = trajectory_digits[-1] == eval_true_digits
print(f"Accuracy finale sur le batch d'évaluation: {eval_correct.float().mean().item():.2%}")

In [ ]:
# Visualisation de la trajectoire (départ, milieu, fin) pour quelques exemples,
# avec la confiance/récompense obtenue à chaque étape affichée (dont celle de départ)
n_show = 5
show_steps = [0, N_STEPS // 2, N_STEPS]

fig, axes = plt.subplots(n_show, len(show_steps), figsize=(len(show_steps) * 1.8, n_show * 1.8))
for row in range(n_show):
    for col, step in enumerate(show_steps):
        img = trajectory_images[step][row, 0].cpu().numpy()
        axes[row, col].imshow(img, cmap="gray")
        axes[row, col].axis("off")

        conf = trajectory_confidences[step][row].item()
        digit = trajectory_digits[step][row].item()
        label = "départ" if step == 0 else f"pas {step}"
        axes[row, col].set_title(f"{label}: {digit} ({conf:.2f})", fontsize=8)

plt.tight_layout()
plt.show()